# 03. MPNN으로 분자 용해도(ESOL) 예측하기

앞 실습에서 만든 **특성 행렬 X 와 인접 행렬 A** 를 그대로 입력으로 사용해
**MPNN(Message Passing Neural Network)** 을 직접 구현하고, 물에 대한 용해도(logS)를 **회귀(Regression)** 로 예측함.

## 전체 흐름
```
ESOL 데이터(SMILES, logS)
   → 분자 그래프 변환 (X, A)          ← 앞 실습 코드
   → 데이터 분할 (Train / Validation / Test)
   → MPNN 모델 (Message → Aggregate → Update  ×3번  → Readout)
   → 학습 → 예측 → 성능 평가(RMSE, MAE, R²) → 시각화(Parity, Residual)
```

## MPNN 한 줄 요약
> 각 원자는 **이웃 원자들로부터 메시지를 받아** 자기 상태를 갱신하고,
> 이 과정을 여러 번 반복한 뒤 **모든 원자 상태를 합쳐** 분자 전체의 값을 예측함

- 논문: Gilmer et al., *Neural Message Passing for Quantum Chemistry*, ICML 2017

## 0. 환경 설정

Google Colab에는 **RDKit만 기본 설치되어 있지 않음** → `pip` 로 먼저 설치함

| 라이브러리 | 용도 |
|------|------|
| `rdkit` | SMILES → 분자 객체, 원자·결합 정보 추출 |
| `torch` | MPNN 모델 구현 및 학습 (Colab 기본 설치) |
| `scikit-learn` | 데이터 분할, 성능 지표 계산 (Colab 기본 설치) |
| `numpy`, `pandas` | 배열 / 표 처리 |
| `matplotlib` | 시각화 |

In [ ]:
# (Colab) RDKit 설치 - 처음 한 번만 실행하면 된다.
!pip install rdkit

from rdkit import Chem                          # SMILES → 분자 객체(Mol)
import numpy as np                              # 배열 처리
import pandas as pd                             # 표(DataFrame) 처리
import matplotlib.pyplot as plt                 # 시각화

import torch                                    # 딥러닝 프레임워크
import torch.nn as nn                           # 신경망 층(Linear 등)

from sklearn.model_selection import train_test_split                           # 데이터 분할
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # 성능 지표

# 실행할 때마다 같은 결과가 나오도록 난수 고정
torch.manual_seed(42)
np.random.seed(42)

---
## 1. ESOL 데이터셋 불러오기

> **[실습] 분자 구조(SMILES)와 실험 용해도(logS) 데이터 불러오기**

- **ESOL (Delaney)** : 1,128개 분자의 **물 용해도** 데이터셋 (MoleculeNet 벤치마크)
- 용해도 단위 : **log(mol/L)** → 값이 클수록 물에 잘 녹음

| 사용할 컬럼 | 의미 | 역할 |
|------|------|------|
| `smiles` | 분자 구조 문자열 | 입력 (그래프로 변환) |
| `measured log solubility in mols per litre` | 실험으로 측정한 logS | 정답(y) |

In [ ]:
# [1] ESOL 데이터셋 불러오기 (DeepChem 공개 데이터)
url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
df = pd.read_csv(url)

# [2] 필요한 컬럼만 남기고 이름을 짧게 바꾸기
df = df[['smiles', 'measured log solubility in mols per litre']]
df.columns = ['smiles', 'logS']

print('분자 수 :', len(df))
display(df.head())

# [3] 정답(logS) 분포 확인
plt.figure(figsize=(6, 4))
plt.hist(df['logS'], bins=30, color='steelblue', edgecolor='black')
plt.xlabel('logS (mol/L)')
plt.ylabel('Count')
plt.title('ESOL solubility distribution')
plt.show()

---
## 2. SMILES → 분자 그래프 (X, A) 변환

> **[실습] 앞 실습의 특성 행렬 / 인접 행렬 코드를 함수로 묶어 모든 분자에 적용하기**

| 이름 | 크기 | 내용 |
|------|------|------|
| `X` (특성 행렬) | (원자 수, 12) | 원소 원-핫 9 + degree + num_Hs + aromatic (앞 실습과 동일) |
| `A` (인접 행렬) | (원자 수, 원자 수) | 결합된 원자 쌍이면 1, 아니면 0 (앞 실습과 동일) |
| `y` (정답) | 1개 값 | 실험 logS |

In [ ]:
atom_list = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'Other']


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)

    # [1] 특성 행렬 X : 원자마다 [원소 원-핫 + degree + num_Hs + aromatic]
    X = []
    for atom in mol.GetAtoms():
        one_hot = [0] * len(atom_list)
        if atom.GetSymbol() in atom_list:
            one_hot[atom_list.index(atom.GetSymbol())] = 1
        else:
            one_hot[-1] = 1
        X.append(one_hot + [atom.GetDegree(), atom.GetTotalNumHs(), int(atom.GetIsAromatic())])

    # [2] 인접 행렬 A : 결합된 원자 쌍 (i, j), (j, i) 를 1로
    num_atoms = mol.GetNumAtoms()
    A = np.zeros((num_atoms, num_atoms))
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        A[i, j] = 1
        A[j, i] = 1

    # [3] PyTorch 텐서로 변환 (모델에 넣기 위해)
    X = torch.tensor(X, dtype=torch.float)
    A = torch.tensor(A, dtype=torch.float)
    return X, A


# 모든 분자를 (X, A, y) 로 변환해 리스트에 저장
dataset = []
for smi, logS in zip(df['smiles'], df['logS']):
    X, A = smiles_to_graph(smi)
    y = torch.tensor([logS], dtype=torch.float)
    dataset.append((X, A, y))

print('변환된 분자 수 :', len(dataset))

# 첫 번째 분자 확인
X, A, y = dataset[0]
print('SMILES :', df['smiles'][0])
print('X 크기 :', tuple(X.shape), '→ (원자 수, 특성 수)')
print('A 크기 :', tuple(A.shape), '→ (원자 수, 원자 수)')
print('y      :', y.item())

---
## 3. 데이터 분할 (Train / Validation / Test)

> **[실습] 학습용 / 검증용 / 평가용 데이터 나누기**

| 데이터 | 비율 | 역할 |
|------|------|------|
| Train | 80% | 모델의 가중치를 **학습** |
| Validation | 10% | 학습 중 성능 확인 (과적합 여부 체크) |
| Test | 10% | 학습에 전혀 쓰지 않은 데이터로 **최종 성능 평가** |

- `train_test_split` 을 두 번 사용함 : 전체 → (Train 80%, 나머지 20%) → 나머지를 (Val 10%, Test 10%)

In [ ]:
# [1] 전체 → Train(80%) / 나머지(20%)
train_data, temp_data = train_test_split(dataset, test_size=0.2, random_state=42)

# [2] 나머지(20%) → Validation(10%) / Test(10%)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print('Train      :', len(train_data))
print('Validation :', len(val_data))
print('Test       :', len(test_data))

---
## 4. MPNN 모델 구현

> **[실습] Message → Aggregate → Update → Readout 을 직접 구현하기**

### 4-1. 인접 행렬 A 로 "이웃 메시지 더하기"

`A @ h` 를 하면 **각 원자 자리에 이웃 원자들의 값이 더해짐** (A 에서 1인 칸만 더해지기 때문)

```
      C0 C1 O2              h            A @ h
C0  [ 0  1  0 ]          [ h0 ]        [ h1      ]   ← C0의 이웃 : C1
C1  [ 1  0  1 ]    @     [ h1 ]   =    [ h0 + h2 ]   ← C1의 이웃 : C0, O2
O2  [ 0  1  0 ]          [ h2 ]        [ h1      ]   ← O2의 이웃 : C1
```

### 4-2. MPNN 의 구성

| 단계 | 의미 | 코드 |
|------|------|------|
| ⓪ Embedding | 원자 특성(12차원)을 hidden 차원으로 변환 | `self.embed(X)` |
| ① Message | 각 원자가 이웃에게 보낼 **메시지** 를 만듦 | `self.message(h)` |
| ② Aggregate | 이웃들이 보낸 메시지를 **모두 더함** | `A @ msg` |
| ③ Update | (내 상태 + 받은 메시지)로 **내 상태를 갱신** | `self.update(...)` |
| ④ Readout | 모든 원자 상태를 **더해서** 분자 하나의 벡터로 | `h.sum(dim=0)` |
| ⑤ Predict | 분자 벡터로 **logS 예측** | `self.predict(h_mol)` |

- ①~③ 을 **3번 반복** → 원자는 3칸 떨어진 원자의 정보까지 알게 됨

In [ ]:
# *************#
# 실습 코드 작성 #
# *************#

---
## 5. 모델 학습

> **[실습] 분자를 하나씩 넣어가며 MPNN 학습하기**

| 설정 | 값 | 의미 |
|------|------|------|
| 손실 함수 | `MSELoss` | (예측 − 정답)² → 작을수록 좋음 |
| 최적화 | `Adam`, lr = 0.001 | 가중치를 손실이 줄어드는 방향으로 조금씩 수정 |
| epochs | 30 | Train 데이터 전체를 몇 번 반복 학습할지 |

**분자 하나에 대한 학습 단계**
```
[1] 예측  →  [2] 손실 계산  →  [3] 기울기 초기화  →  [4] 기울기 계산(backward)  →  [5] 가중치 수정(step)
```

In [ ]:
def evaluate(model, data):
    # 데이터 전체에 대한 RMSE 계산
    errors = []
    with torch.no_grad():                       # 평가할 때는 기울기 계산 X
        for X, A, y in data:
            y_hat = model(X, A)
            errors.append((y_hat - y).item() ** 2)
    return np.sqrt(np.mean(errors))



# *************#
# 실습 코드 작성 #
# *************#

In [ ]:
# 학습 곡선 : epoch 에 따른 Train / Validation RMSE
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(train_history) + 1), train_history, marker='o', label='Train')
plt.plot(range(1, len(val_history) + 1), val_history, marker='o', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('RMSE (logS)')
plt.title('Learning curve')
plt.legend()
plt.show()

---
## 6. 예측

> **[실습] 학습된 모델로 Train / Test 분자의 logS 예측하기**

- Test 데이터는 학습에 쓰지 않았으므로 **처음 보는 분자에 대한 성능** 을 보여줌

In [ ]:
# *************#
# 실습 코드 작성 #
# *************#

---
## 7. 성능 평가

> **[실습] 회귀 성능 지표 계산하기**

| 지표 | 의미 | 좋은 값 |
|------|------|------|
| RMSE | 평균적인 오차 크기 (큰 오차에 민감) | **작을수록** 좋음 |
| MAE | 오차 절댓값의 평균 | **작을수록** 좋음 |
| R² | 정답의 변동을 얼마나 설명하는지 | **1에 가까울수록** 좋음 |

- Train 성능만 좋고 Test 성능이 많이 나쁘면 **과적합(Overfitting)** 을 의심함

In [ ]:
for name, y_true, y_pred in [('Train', y_train, pred_train), ('Test', y_test, pred_test)]:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name:5s} | RMSE {rmse:.3f} | MAE {mae:.3f} | R² {r2:.3f}')

---
## 8. 결과 시각화

> **[실습] Parity plot / Residual plot / Residual distribution 그리기**

| 그림 | x축 | y축 | 좋은 모델이라면 |
|------|------|------|------|
| ① Parity plot | 실제 logS | 예측 logS | 점들이 **대각선(y = x)** 위에 모여 있음 |
| ② Residual plot | 예측 logS | 잔차 (실제 − 예측) | 점들이 **0 주변에 고르게** 퍼져 있음 |
| ③ Residual distribution | 잔차 | 개수 | **0을 중심으로 좁고 대칭** 인 종 모양 |

In [ ]:
res_train = y_train - pred_train          # Train 잔차
res_test = y_test - pred_test             # Test 잔차

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ① Parity plot : 실제 vs 예측
ax = axes[0]
ax.scatter(y_train, pred_train, s=15, alpha=0.4, color='gray', label='Train')
ax.scatter(y_test, pred_test, s=25, color='tomato', edgecolors='black', label='Test')
ax.plot([-12, 2], [-12, 2], 'k--', label='y = x')       # 완벽한 예측선
ax.set_xlabel('Measured logS')
ax.set_ylabel('Predicted logS')
ax.set_title('Parity plot')
ax.legend()

# ② Residual plot : 예측값 vs 잔차
ax = axes[1]
ax.scatter(pred_train, res_train, s=15, alpha=0.4, color='gray', label='Train')
ax.scatter(pred_test, res_test, s=25, color='tomato', edgecolors='black', label='Test')
ax.axhline(0, color='black', linestyle='--')            # 잔차 = 0 기준선
ax.set_xlabel('Predicted logS')
ax.set_ylabel('Residual (Measured - Predicted)')
ax.set_title('Residual plot')
ax.legend()

# ③ Residual distribution : 잔차 히스토그램
ax = axes[2]
ax.hist(res_train, bins=30, alpha=0.5, color='gray', edgecolor='black', density=True, label='Train')
ax.hist(res_test, bins=30, alpha=0.6, color='tomato', edgecolor='black', density=True, label='Test')
ax.axvline(0, color='black', linestyle='--')
ax.set_xlabel('Residual (Measured - Predicted)')
ax.set_ylabel('Density')
ax.set_title('Residual distribution')
ax.legend()

plt.tight_layout()
plt.show()

---
## 9. 정리

| 단계 | 한 일 |
|------|------|
| 그래프 변환 | SMILES → 특성 행렬 `X`, 인접 행렬 `A` (앞 실습 코드 재사용) |
| 데이터 분할 | Train 80% / Validation 10% / Test 10% |
| MPNN | **Message**(`message(h)`) → **Aggregate**(`A @ msg`) → **Update**(`update([h, m])`) × 3회 → **Readout**(`sum`) |
| 학습 | MSE 손실 + Adam, 분자 하나씩 학습 |
| 평가 | RMSE / MAE / R², Parity · Residual plot 으로 확인 |

**더 해보기 🖐️**
- `forward` 의 반복 횟수 `range(3)` 를 1, 5 로 바꾸면 성능이 어떻게 달라지는지 비교해 보기
- Readout 을 합 `h.sum(dim=0)` 대신 평균 `h.mean(dim=0)` 으로 바꿔 보기
- `hidden_dim` 이나 epoch 수를 바꿔 보기